!apt-get install -y openjdk-11-jdk   -qq

In [58]:
import pandas as pd
from owlready2 import *

## Загрузка данных

In [59]:
# noinspection PyArgumentList
knowledge_df = pd.read_csv('ontology_data/knowledge.csv')
# noinspection PyArgumentList
competencies_df = pd.read_csv('ontology_data/competencies.csv')
# noinspection PyArgumentList
competency_knowledge_df = pd.read_csv('ontology_data/competency_knowledge.csv')
# noinspection PyArgumentList
positions_df = pd.read_csv('ontology_data/positions.csv')
# noinspection PyArgumentList
position_competency_df = pd.read_csv('ontology_data/position_competency.csv')
# noinspection PyArgumentList
employees_df = pd.read_csv('ontology_data/employees.csv')


In [60]:
knowledge_df

,knowledge_id,knowledge_name
0,1,Математическая статистика
1,2,Линейная алгебра
2,3,Теория вероятностей
3,4,Python программирование
4,5,"Библиотеки ML (scikit-learn, TensorFlow, PyTorch)"
5,6,Обработка естественного языка (NLP)
6,7,Компьютерное зрение
7,8,"Работа с большими данными (Spark, Hadoop)"
8,9,SQL и базы данных
9,10,Визуализация данных


# Создание новой онтологии

## Итерация 1.
Создадим знания и компетенции

In [61]:
# Создаем онтологию
onto = get_ontology("http://test.org/ml_specialist_v2.owl")

with onto:
    class Knowledge(Thing):
        pass


    class Competence(Thing):
        pass


    class requires_knowledge(ObjectProperty):
        domain = [Competence]
        range = [Knowledge]


    class Name(DataProperty):
        domain = [Thing]
        range = [str]

### Создание индивидуальностей для знаний и компетенций



In [62]:
def create_knowledge(knowledge_df):
    with onto:
        knowledge_instances = {}
        for _, row in knowledge_df.iterrows():
            knowledge = Knowledge(f"знание_{row['knowledge_id']}")
            knowledge.name = [row['knowledge_name']]
            knowledge_instances[row['knowledge_id']] = knowledge
        return knowledge_instances


def create_competencies(competencies_df):
    with onto:
        competency_instances = {}
        for _, row in competencies_df.iterrows():
            competency = Competence(f"компетенция_{row['competency_id']}")
            competency.name = [row['competency_name']]
            competency_instances[row['competency_id']] = competency
        return competency_instances


In [63]:
knowledges = create_knowledge(knowledge_df)

In [64]:
knowledges

{1: ml_specialist_v2.['Математическая статистика'],
 2: ml_specialist_v2.['Линейная алгебра'],
 3: ml_specialist_v2.['Теория вероятностей'],
 4: ml_specialist_v2.['Python программирование'],
 5: ml_specialist_v2.['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)'],
 6: ml_specialist_v2.['Обработка естественного языка (NLP)'],
 7: ml_specialist_v2.['Компьютерное зрение'],
 8: ml_specialist_v2.['Работа с большими данными (Spark, Hadoop)'],
 9: ml_specialist_v2.['SQL и базы данных'],
 10: ml_specialist_v2.['Визуализация данных']}

In [65]:
for id, item in knowledges.items():
    print(item.name[0])

Математическая статистика
Линейная алгебра
Теория вероятностей
Python программирование
Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Обработка естественного языка (NLP)
Компьютерное зрение
Работа с большими данными (Spark, Hadoop)
SQL и базы данных
Визуализация данных


In [66]:
competencies = create_competencies(competencies_df)
for id, item in competencies.items():
    print(item.name[0])

Математическая подготовка
Программирование и алгоритмы
Машинное обучение
Глубокое обучение
Обработка и анализ данных
Работа с большими данными
Визуализация и представление данных
Развертывание ML-моделей


### Создание отношений между знаниями и компетенциями

In [67]:
# Связываем компетенции со знаниями
def relate_competencies_knowledge(df, competency_instances, knowledge_instances):
    with onto:
        for _, row in df.iterrows():
            comp = competency_instances[row['competency_id']]
            know = knowledge_instances[row['knowledge_id']]

            # Вместо 'onto.requires_knowledge' используем само имя класса свойства
            # Owlready2 поймет, что ты хочешь добавить 'know' к списку связей 'comp'
            if know not in comp.requires_knowledge:
                comp.requires_knowledge.append(know)


In [68]:
relate_competencies_knowledge(competency_knowledge_df, competencies, knowledges)

Выведем связи между компенциями и знаниями

In [69]:
def print_competency_knowledge():
    print("\n" + "=" * 60)
    print("Связь знаний и компетенций")
    print("=" * 60)

    for competency_id, competency in competencies.items():
        comp_name = competency.name[0]
        print(f"\n{comp_name} требует знания:")

        required_knowledge = list(competency.requires_knowledge)
        for know in required_knowledge:
            print(f"  - {know.name[0]}")


print_competency_knowledge()



Связь знаний и компетенций

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

# Итерация 2

Добавление новых концептов в онтологию


In [70]:
with onto:
    class Specialist(Thing):
        pass


    class Job(Thing):
        pass


    class has_knowledge(ObjectProperty):
        domain = [Specialist]
        range = [Knowledge]


    class has_competence(ObjectProperty):
        domain = [Specialist]
        range = [Competence]


    class can_hold_position(ObjectProperty):
        domain = [Specialist]
        range = [Job]


    class requires_competence(ObjectProperty):
        domain = [Job]
        range = [Competence]


    class EmployeeName(DataProperty):
        domain = [Specialist]
        range = [str]

In [71]:
employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'].apply(list)

employee_id  full_name                     
1            Иванов Алексей Сергеевич           [1, 4, 5]
2            Петрова Мария Владимировна         [1, 4, 5]
3            Сидоров Дмитрий Петрович           [1, 2, 9]
4            Козлова Анна Игоревна              [5, 6, 7]
5            Федоров Максим Андреевич           [5, 6, 7]
6            Николаева Екатерина Дмитриевна    [4, 5, 10]
7            Орлов Сергей Викторович            [1, 2, 3]
Name: knowledge_id, dtype: object

### Создание индивидуальностей для сотрудников и их отношений с знаниями


In [72]:
def create_employees(employees_df, knowledge_instances):
    employees_df = employees_df.drop_duplicates()
    employee_knowledge = employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'] \
        .apply(lambda x: list(set(x))).reset_index()

    employee_instances = {}
    for _, row in employee_knowledge.iterrows():
        # создадим сотрудника
        employee = Specialist(f"сотрудник_{row['employee_id']}")
        employee.EmployeeName = [row['full_name']]
        employee_instances[row['employee_id']] = employee

        # добавим знания сотруднику
        for knowledge_id in row['knowledge_id']:
            if knowledge_id in knowledge_instances:
                employee.has_knowledge.append(knowledge_instances[knowledge_id])

    return employee_instances

In [73]:
employees = create_employees(employees_df, knowledges)

In [74]:
def print_employee_mindmap():
    print("\n" + "=" * 60)
    print("Карта знаний сотрудников")
    print("=" * 60)
    for employee_id, employee in employees.items():
        employee_name = employee.EmployeeName[0] if employee.EmployeeName else f"Сотрудник {employee_id}"

        print(f"\n{employee_name}:")

        # Получаем список знаний сотрудника
        knowledge_list = list(employee.has_knowledge)

        if knowledge_list:
            print("Знания:")
            for knowledge in knowledge_list:
                know_name = knowledge.Name[0] if knowledge.Name else knowledge.name
                print(f"  - {know_name}")
        else:
            print("Знания: нет")


print_employee_mindmap()


Карта знаний сотрудников

Иванов Алексей Сергеевич:
Знания:
  - ['Математическая статистика']
  - ['Python программирование']
  - ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)']

Петрова Мария Владимировна:
Знания:
  - ['Математическая статистика']
  - ['Python программирование']
  - ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)']

Сидоров Дмитрий Петрович:
Знания:
  - ['Математическая статистика']
  - ['Линейная алгебра']
  - ['SQL и базы данных']

Козлова Анна Игоревна:
Знания:
  - ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)']
  - ['Обработка естественного языка (NLP)']
  - ['Компьютерное зрение']

Федоров Максим Андреевич:
Знания:
  - ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)']
  - ['Обработка естественного языка (NLP)']
  - ['Компьютерное зрение']

Николаева Екатерина Дмитриевна:
Знания:
  - ['Визуализация данных']
  - ['Python программирование']
  - ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)']

Орлов Сергей Викторович:
Знания:
  - ['Математи

### Создание индивидуальностей для должностей и из связей с компетенциями


In [75]:
# Создаем экземпляры должностей
def create_position_instances(positions_df):
    position_instances = {}
    for _, row in positions_df.iterrows():
        position = Job(f"должность_{row['position_id']}")
        position.name = row['position_name']
        position_instances[row['position_id']] = position
    return position_instances

In [76]:
positions = create_position_instances(positions_df)
positions

{1: ml_specialist_v2.ML Engineer,
 2: ml_specialist_v2.Data Scientist,
 3: ml_specialist_v2.Data Analyst,
 4: ml_specialist_v2.NLP Engineer,
 5: ml_specialist_v2.Computer Vision Engineer}

In [77]:
# Устанавливаем связи между должностями и компетенциями
def set_position_competency_relations(position_competency_df, position_instances, competency_instances):
    for _, row in position_competency_df.iterrows():
        position = position_instances[row['position_id']]
        competency = competency_instances[row['competency_id']]
        position.requires_competence.append(competency)

In [78]:
set_position_competency_relations(position_competency_df, positions, competencies)

In [79]:
# сюда вставить код для выполнения самостоятельных заданий
positions[4].requires_competence.append(competencies[4])



## Выведем требования к должностям

In [80]:
# Проверим требования должностей
def print_position_require():
    print("\n" + "=" * 60)
    print("ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ")
    print("=" * 60)

    for position_id, position in positions.items():
        pos_name = position.name
        print(f"\n{pos_name} требует компетенции:")

        required_competencies = list(position.requires_competence)
        if required_competencies:
            for comp in required_competencies:
                print(f"  - {comp.name[0]}")
        else:
            print("  - нет требований")


print_position_require()


ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ

ML Engineer требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Scientist требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Analyst требует компетенции:
  - Обработка и анализ данных

NLP Engineer требует компетенции:
  - Программирование и алгоритмы
  - Глубокое обучение

Computer Vision Engineer требует компетенции:
  - Машинное обучение


## Создание аксиом для онтологии

 <b>Аксиома 1:</b> Если специалист обладает всеми требуемыми знаниями для компетенции, то у него есть эта компетенция

 <b>Аксиома 2:</b> Если у специалиста есть требуемые компетенции для должности, то он может занимать эту должность

  <b> Аксиома 3:</b> (для проверки корректности онтологии) Если специалист занимает должность, то у него должны быть соответствующие компетенции
      


In [81]:
def define_rules():
    # Аксиома 1:
    with onto:
        class HasCompenceForKnowledge(Specialist >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой компетенции проверяем, есть ли у специалиста все необходимые знания
                for competency in onto.Competence.instances():
                    required_knowledge = list(competency.requires_knowledge)
                    specialist_knowledge = list(specialist.has_knowledge)

                    # Check that all required knowledge is possessed by the specialist
                    if all(knowledge in specialist_knowledge for knowledge in required_knowledge):
                        if competency not in specialist.has_competence:
                            specialist.has_competence.append(competency)

                return True
        # Аксиома 2:
        class CanHoldJobForCompetencies(Specialist >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой должности проверяем, есть ли у специалиста все необходимые компетенции
                for position in onto.Job.instances():
                    required_competencies = list(position.requires_competence)
                    specialist_competencies = list(specialist.has_competence)

                    # Check that all required competencies are possessed by the specialist
                    if all(competency in specialist_competencies for competency in required_competencies):
                        if position not in specialist.can_hold_position:
                            specialist.can_hold_position.append(position)

                return True

        # Аксиома 3:
        class CheckCompetenciesForJob(Specialist >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                positions = list(specialist.can_hold_position)
                competencies = list(specialist.has_competence)

                for position in positions:
                    required_competencies = list(position.requires_competence)
                    if not all(comp in competencies for comp in required_competencies):
                        print(f"Внимание: {specialist.name} не имеет все компетенции для {position.name}")

                return True

    return HasCompenceForKnowledge(), CanHoldJobForCompetencies(), CheckCompetenciesForJob()

In [82]:
rule1, rule2, rule3 = define_rules()

In [83]:
# Проверим связи знаний и компетенций
print_competency_knowledge()


Связь знаний и компетенций

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

In [84]:
onto.Specialist.instances()

[ml_specialist_v2.сотрудник_1,
 ml_specialist_v2.сотрудник_2,
 ml_specialist_v2.сотрудник_3,
 ml_specialist_v2.сотрудник_4,
 ml_specialist_v2.сотрудник_5,
 ml_specialist_v2.сотрудник_6,
 ml_specialist_v2.сотрудник_7]

In [85]:
for employee in onto.Specialist.instances():
    rule1(employee)  # Применяем правило для компетенций
    rule2(employee)  # Применяем правило для должностей
    rule3(employee)  # Проверяем соответствие

## Проведение анализа данных

In [86]:
def print_position_for_employee():
    print("\n" + "=" * 60)
    print("АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ")
    print("=" * 60)

    for employee in onto.Specialist.instances():
        print(f"\nСотрудник: {employee.EmployeeName[0]}")

        print("Знания:")
        for knowledge in employee.has_knowledge:
            print(f"  - {knowledge.name[0]}")

        print("Компетенции:")
        for competency in employee.has_competence:
            print(f"  - {competency.name[0]}")

        print("Может занимать должности:")
        for position in employee.can_hold_position:
            print(f"  - {position.name}")

        print("-" * 30)


print_position_for_employee()


АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ

Сотрудник: Иванов Алексей Сергеевич
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - Computer Vision Engineer
------------------------------

Сотрудник: Петрова Мария Владимировна
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - Computer Vision Engineer
------------------------------

Сотрудник: Сидоров Дмитрий Петрович
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных
Компетенции:
  - Обработка и анализ данных
Может занимать должности:
  - Data Analyst
-

In [87]:
# Убедимся, что у сотрудников достаточно знаний для получения компетенций
def check_employee_competencies(employee_instances, competency_instances):
    """Проверяем, какие компетенции могут получить сотрудники"""
    print("\n" + "=" * 60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ")
    print("=" * 60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for competency_id, competency in competency_instances.items():
            comp_name = competency.название[0]
            required_knowledge = set(competency.требует_знание)
            employee_knowledge = set(employee.обладает_знанием)

            # Проверяем, есть ли у сотрудника все необходимые знания
            if required_knowledge.issubset(employee_knowledge):
                print(f"  Может получить компетенцию: {comp_name}")
                # Присваиваем компетенцию
                if competency not in employee.обладает_компетенцией:
                    employee.обладает_компетенцией.append(competency)
            else:
                missing_knowledge = required_knowledge - employee_knowledge
                if missing_knowledge:
                    print(f"  Не хватает для '{comp_name}':")
                    for know in missing_knowledge:
                        print(f"      - {know.название[0]}")


In [88]:
# Убедимся, что у сотрудников достаточно знаний для получения компетенций
def check_employee_competencies(employee_instances, competency_instances):
    """Проверяем, какие компетенции могут получить сотрудники"""
    print("\n" + "=" * 60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ")
    print("=" * 60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.EmployeeName[0]
        print(f"\n{employee_name}:")

        for competency_id, competency in competency_instances.items():
            comp_name = competency.name[0]
            required_knowledge = set(competency.requires_knowledge)
            employee_knowledge = set(employee.has_knowledge)

            # Проверяем, есть ли у сотрудника все необходимые знания
            if required_knowledge.issubset(employee_knowledge):
                print(f"  Может получить компетенцию: {comp_name}")
                # Присваиваем компетенцию
                if competency not in employee.has_competence:
                    employee.has_competence.append(competency)
            else:
                missing_knowledge = required_knowledge - employee_knowledge
                if missing_knowledge:
                    print(f"  Не хватает для '{comp_name}':")
                    for know in missing_knowledge:
                        print(f"      - {know.name[0]}")

check_employee_competencies(employees, competencies)


ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      - Компьютерное зрение
      - Обработка естественного языка (NLP)
  Не хватает для 'Обработка и анализ данных':
      - SQL и базы данных
  Не хватает для 'Работа с большими данными':
      - Работа с большими данными (Spark, Hadoop)
      - SQL и базы данных
  Не хватает для 'Визуализация и представление данных':
      - Визуализация данных
  Может получить компетенцию: Развертывание ML-моделей

Петрова Мария Владимировна:
  Не хватает для 'Математическая подготовка':
      - Теория вероятностей
      - Линейная алгебра
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      -

# Задание 1:  Добавление необходимой компетенции в должность

Добавить требование, что компетениция NLP Enginner содержит знание по глубокому обучению.

<B>Важно, что добавление в онтологию новых данных в Питоне должно сопровождаться созданием новой онтологии.</B>


In [89]:
# 1. Находим нужную должность и компетенцию и связываем их
# NLP Engineer = positions[4]
# Глубокое обучение = competencies[4]

# Используем свойство requires_competence
if competencies[4] not in positions[4].requires_competence:
    positions[4].requires_competence.append(competencies[4])

print(f"{positions[4].name} добавлена компетенция {competencies[4].name[0]}")

NLP Engineer добавлена компетенция Глубокое обучение


## Задание 2:
Добавьте требование в онтологию: должность Data Scientist требует компетенции "Математическая подготовка"

In [90]:
#  Находим data scientist и компетенцию математическую подготовку
data_scientist_job = positions[2]
print(data_scientist_job)
math_competency = competencies[1]
print(math_competency)
# 2. Добавляем компетенцию в список требований должности
if math_competency not in data_scientist_job.requires_competence:
    data_scientist_job.requires_competence.append(math_competency)


print(f"для {data_scientist_job.name} теперь требуется {math_competency.name[0]}")

ml_specialist_v2.Data Scientist
ml_specialist_v2.['Математическая подготовка']
для Data Scientist теперь требуется Математическая подготовка


## Задание 3:
Добавьте сотруднику Федорову Максиму Андреевичу необходимые знания, чтобы он мог претендовать на должность Computer Vision Engineer

In [91]:
# находим сотрудника и должность
target_employee = employees[5]
print(*target_employee.EmployeeName)
target_job = positions[5]
print(target_job)


# Собираем все знания, которые нужны для этой должности через компетенции
required_knowledges = set()
for comp in target_job.requires_competence:
    for know in comp.requires_knowledge:
        required_knowledges.add(know)

# смотрим каких знаний сейчас нет вычитание множеств
current_knowledges = set(target_employee.has_knowledge)
missing_knowledges = required_knowledges - current_knowledges

# доблавяем знания которых нет
if missing_knowledges:
    print("\nДобавлены новые знания:")
    for know in missing_knowledges:
        target_employee.has_knowledge.append(know)
        print(f"  + {know.name[0]}")
else:
    print("\nУ сотрудника есть все знания для этой должности.")

# Применяем аксиомы, чтобы онтология "поняла", что теперь он получил нужные компетенции и может занять должность
rule1(target_employee)  # Выдаст компетенции на основе новых знаний
rule2(target_employee)  # Выдаст должность на основе новых компетенций

print(f"\n{target_employee.EmployeeName[0]} может занимать должности:")
for pos in target_employee.can_hold_position:
    print(f"  - {pos.name}")

Федоров Максим Андреевич
ml_specialist_v2.Computer Vision Engineer

Добавлены новые знания:
  + Python программирование
  + Математическая статистика

Федоров Максим Андреевич может занимать должности:
  - ML Engineer
  - NLP Engineer
  - Computer Vision Engineer


## Задание 4:
Добавьте требование в онтологию нового сотрудника, укажите в качестве имя_сотрудника ваше ФИО. Добавьте знания для данного сотрудника, чтобы он смог претендовать на все должности текущей онтологии

In [92]:
# создаем нового сотрудника
new_employee_id = 111222333  # Берем большой id чтобы не конфликтовал
new_employee = Specialist(f"сотрудник_{new_employee_id}")
new_employee.EmployeeName = ["Леушкин Максим Юрьевич"]

employees[new_employee_id] = new_employee

# Собираем все знания для всех должностей
all_required_knowledges = set()
for job in onto.Job.instances():
    for comp in job.requires_competence:
        for know in comp.requires_knowledge:
            all_required_knowledges.add(know)

# даем эти знания новому сотруднику
for know in all_required_knowledges:
    new_employee.has_knowledge.append(know)

print(f"загружено уникальных знаний: {len(all_required_knowledges)}")

# применяем аксиомы
rule1(new_employee)  # Система выдает компетенции на основе полученных знаний
rule2(new_employee)  # Система выдает должности на основе полученных компетенций

# 5. Выводим результат: проверяем, на какие должности теперь можно претендовать
print(f"\n{new_employee.EmployeeName[0]} может занимать должности:")
for pos in new_employee.can_hold_position:
    print(f"  - {pos.name}")

загружено уникальных знаний: 8

Леушкин Максим Юрьевич может занимать должности:
  - ML Engineer
  - Data Scientist
  - Data Analyst
  - NLP Engineer
  - Computer Vision Engineer
